# Advanced rendering

Since 3.0.0 K3D ships a second renderer. `plot.renderer = 'advanced'` switches the
scene to image-based lighting: all light comes from an environment map, materials are
physically based (`roughness` / `metalness`), and the image can go through filmic tone
mapping. Everything below keeps working in the default `'simple'` renderer - it just
lights the scene with the classic light rig instead.

In [ ]:
import k3d
import numpy as np
from numpy import sin, cos

T = 1.618033988749895
r = 4.77
grid = np.linspace(-r, r, 77, dtype=np.float32)
x, y, z = np.meshgrid(grid, grid, grid, indexing='ij')

p = 2 - (cos(x + T * y) + cos(x - T * y) + cos(y + T * z)
         + cos(y - T * z) + cos(z - T * x) + cos(z + T * x))
a = (sin(x + T * y) + sin(x - T * y) + sin(y + T * z)
     + cos(y - T * z) + sin(z - T * x) + cos(z + T * x))

iso = k3d.marching_cubes(p, attribute=a, level=0.0,
                         color_map=k3d.matplotlib_color_maps.Inferno,
                         flat_shading=False, roughness=0.25)

plot = k3d.plot(renderer='advanced', environment='studio')
plot += iso
plot.display()

## Environments

Three procedural presets travel as plain names: `'neutral'` (default), `'studio'` and
`'outdoor'`. `environment_rotation` spins the map around the scene's up axis.

In [ ]:
plot.environment = 'outdoor'
plot.environment_rotation = np.pi / 3

## Photographic environments

A small catalog of Poly Haven HDRIs (CC0) ships with the package, and any `(H, W, 3)`
float32 equirect array works as well. Every map is energy-normalised, so the
environment carries the *shape* of the light while `plot.lighting` stays the exposure
knob.

In [ ]:
import k3d.environments

print(k3d.environments.available())

plot.environment = 'burnt_warehouse'

## Your own environment

Any `(H, W, 3)` float32 equirectangular array acts as the light source - row 0
is the sky, values are linear radiance (HDR welcome, the map is
energy-normalised). A procedural dusk with a warm horizon glow:

In [ ]:
elevation = np.linspace(1.0, 0.0, 128, dtype=np.float32)[:, None]
azimuth = np.linspace(0.0, 1.0, 256, dtype=np.float32)[None, :]

sky = np.stack([0.2 + 0.1 * elevation,
                0.25 + 0.2 * elevation,
                0.5 + 0.5 * elevation], axis=-1)
glow = (np.exp(-((elevation - 0.5) / 0.1) ** 2)
        * np.exp(-((azimuth - 0.35) / 0.15) ** 2))[..., None]
dusk = (sky + glow * np.array([3.0, 1.2, 0.3], dtype=np.float32)).astype(np.float32)

plot.environment = dusk

## Materials

`roughness` runs left to right, the back row is fully metallic copper. Metals only
reflect their environment, so this is where the advanced renderer earns its keep.

In [ ]:
materials = k3d.plot(renderer='advanced', environment='studio', grid_visible=False)

for i, roughness in enumerate(np.linspace(0.05, 0.95, 6)):
    for j, metalness in enumerate([0.0, 1.0]):
        materials += k3d.points([[1.8 * i, 0.0, 1.8 * j]], point_size=1.5,
                                shader='mesh', mesh_detail=3, color=0xB87333,
                                roughness=float(roughness), metalness=float(metalness))

materials.display()

## Volumetric data under the same light

Volume and MIP read their diffuse light from the environment's spherical harmonics and
one directional light distilled from it - a directional HDRI like `burnt_warehouse`
gives volumetric data real modelling, consistent with every other object in the scene.

In [ ]:
g = np.mgrid[-1:1:64j, -1:1:64j, -1:1:64j].astype(np.float32)
blob = sum(np.exp(-14.0 * ((g[0] - cx) ** 2 + (g[1] - cy) ** 2 + (g[2] - cz) ** 2))
           for cx, cy, cz in [(-0.3, -0.2, 0.0), (0.35, 0.1, 0.15), (0.0, 0.3, -0.25)])

volumetric = k3d.plot(renderer='advanced', environment='burnt_warehouse')
volumetric += k3d.volume(blob.astype(np.float32),
                         color_map=k3d.matplotlib_color_maps.YlOrRd,
                         alpha_coef=18.0)
volumetric.display()

## Tone mapping

Bright HDR highlights can be compressed with a filmic curve: `'agx'` or `'aces'`
(default `'none'` keeps the historical response).

In [ ]:
volumetric.tone_mapping = 'agx'

## Notes

- `shininess` is gone since 3.0.0; the equivalent is
  `roughness = sqrt(2 / (shininess + 2))`, plus `metalness` as a new degree of freedom.
- Both renderers support screenshots, snapshots and headless rendering.
- Switching back is one assignment: `plot.renderer = 'simple'`.